# Group 9 · Retrain với BRFSS 2025

Notebook chạy pipeline retrain (`retrain_pipeline.py`) với dữ liệu BRFSS 2025: model v1.0.0 (train trên BRFSS 2015) được so với model train lại có thêm dữ liệu 2025.

**Cách dùng:** *Runtime › Run all*. Tổng thời gian khoảng 15–25 phút (phần lâu nhất là tải dữ liệu và train).

- Không cần upload gì nếu repo GitHub đang public. Nếu repo private: upload file zip của repo (Code › Download ZIP) vào `/content` trước khi chạy.
- Notebook **không** thay model đang chạy trong app (không dùng `--promote`). Đây là lần chạy để lấy kết quả.

In [ ]:
# 1 · Lấy code của nhóm
import os, glob, subprocess, zipfile, shutil

REPO_URL = "https://github.com/phkh4nh/group9-diabetes-screening.git"
APP = "/content/repo/diabetes_app"

if not os.path.isdir(APP):
    r = subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], capture_output=True, text=True)
    if r.returncode != 0:
        zips = glob.glob("/content/group9-diabetes-screening*.zip")
        if not zips:
            raise RuntimeError("Không clone được repo (có thể đang private). Upload file zip của repo vào /content rồi chạy lại ô này.")
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall("/content/_repo_zip")
        os.makedirs("/content/repo", exist_ok=True)
        shutil.move(glob.glob("/content/_repo_zip/*/diabetes_app")[0], APP)

os.chdir(APP)
assert os.path.isfile("retrain_pipeline.py"), "Repo chưa có retrain_pipeline.py"
print("OK ·", APP)

In [ ]:
# 2 · Cài đúng phiên bản thư viện mà model của nhóm dùng
!pip install -q "xgboost==3.4.1" "scikit-learn==1.6.1" "shap==0.50.0" ucimlrepo
!python -c "import xgboost, sklearn; print('xgboost', xgboost.__version__, '· scikit-learn', sklearn.__version__)"

In [ ]:
# 3 · Dữ liệu 2015: lấy từ UCI (id 891), đúng nguồn notebook phát triển model đã dùng
import pandas as pd

os.makedirs("data", exist_ok=True)
HIST = "data/diabetes_binary_health_indicators_BRFSS2015.csv"
if not os.path.exists(HIST):
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=891)
    pd.concat([ds.data.features, ds.data.targets], axis=1).to_csv(HIST, index=False)

h = pd.read_csv(HIST)
print(f"BRFSS 2015: {h.shape[0]:,} dòng × {h.shape[1]} cột · tỉ lệ dương tính {h.Diabetes_binary.mean():.2%}")
assert h.shape == (253680, 22), "Sai dữ liệu 2015 (phải là 253,680 × 22)"

In [ ]:
# 4 · Lấy file BRFSS 2025: dùng file đã upload nếu có, nếu không thì thử tải từ CDC
XPT_URL = "https://www.cdc.gov/brfss/annual_data/2025/files/LLCP2025XPT.zip"

def find_xpt():
    return glob.glob("/content/LLCP2025.XPT*") + glob.glob("/LLCP2025.XPT*")

def good_zips():
    for z in glob.glob("/content/*.zip") + glob.glob("/*.zip"):
        if "2025" in os.path.basename(z) and not zipfile.is_zipfile(z):
            os.remove(z)                      # file hỏng (CDC chặn trả về trang web thay vì zip)
    return [z for z in glob.glob("/content/*2025*.zip") + glob.glob("/*2025*.zip") if zipfile.is_zipfile(z)]

if not find_xpt() and not good_zips():
    subprocess.run(["curl", "-sSL", "-A", "Mozilla/5.0", "-o", "/content/LLCP2025XPT.zip", XPT_URL])
if not find_xpt():
    zips = good_zips()
    if not zips:
        raise RuntimeError("CDC chặn tải tự động. Tải 'LLCP2025XPT.zip' (64 MB) về máy, "
                           "upload vào /content (không cần giải nén) rồi chạy lại ô này.")
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/content")
XPT = find_xpt()[0]
print(f"{XPT!r} · {os.path.getsize(XPT) / 1e9:.2f} GB")

In [ ]:
# 5 · Recode BRFSS 2025 về đúng 21 biến + Diabetes_binary như file 2015
import json
import numpy as np

YES1_NO0 = {1: 1, 2: 0}   # câu hỏi gốc: 1 = có, 2 = không
NO0_YES1 = {1: 0, 2: 1}   # biến tính sẵn kiểu "risk factor": 1 = không, 2 = có
SRC = ["DIABETE4", "_RFHYPE6", "TOLDHI3", "_CHOLCH3", "_BMI5", "SMOKE100", "CVDSTRK3", "_MICHD",
       "_TOTINDA", "_FRTLT1A", "_VEGLT1A", "_RFDRHV9", "_HLTHPL2", "MEDCOST1", "GENHLTH",
       "MENTHLTH", "PHYSHLTH", "DIFFWALK", "_SEX", "_AGEG5YR", "EDUCA", "INCOME3"]


def recode(raw, schema):
    """BRFSS 2025 (mã gốc CDC) -> 21 biến + Diabetes_binary, giống mã hoá file 2015 (UCI 891)."""
    d = pd.DataFrame(index=raw.index)
    d["Diabetes_binary"] = raw["DIABETE4"].map({1: 1, 2: 0, 3: 0, 4: 0})  # prediabetes = 0, giống file 2015
    d["HighBP"] = raw["_RFHYPE6"].map(NO0_YES1)
    d["HighChol"] = raw["TOLDHI3"].map(YES1_NO0)
    d["CholCheck"] = raw["_CHOLCH3"].map({1: 1, 2: 0, 3: 0})
    d["BMI"] = (raw["_BMI5"] / 100).round()
    d["Smoker"] = raw["SMOKE100"].map(YES1_NO0)
    d["Stroke"] = raw["CVDSTRK3"].map(YES1_NO0)
    d["HeartDiseaseorAttack"] = raw["_MICHD"].map(YES1_NO0)
    d["PhysActivity"] = raw["_TOTINDA"].map(YES1_NO0)
    d["Fruits"] = raw["_FRTLT1A"].map(YES1_NO0)
    d["Veggies"] = raw["_VEGLT1A"].map(YES1_NO0)
    d["HvyAlcoholConsump"] = raw["_RFDRHV9"].map(NO0_YES1)
    d["AnyHealthcare"] = raw["_HLTHPL2"].map(YES1_NO0)
    d["NoDocbcCost"] = raw["MEDCOST1"].map(YES1_NO0)
    d["GenHlth"] = raw["GENHLTH"].where(raw["GENHLTH"].between(1, 5))
    for col, src in (("MentHlth", "MENTHLTH"), ("PhysHlth", "PHYSHLTH")):
        v = raw[src].replace(88, 0)                       # 88 = "không ngày nào"
        d[col] = v.where(v.between(0, 30))
    d["DiffWalk"] = raw["DIFFWALK"].map(YES1_NO0)
    d["Sex"] = raw["_SEX"].map({1: 1, 2: 0})                # 1 = nam, 0 = nữ
    d["Age"] = raw["_AGEG5YR"].where(raw["_AGEG5YR"].between(1, 13))
    d["Education"] = raw["EDUCA"].where(raw["EDUCA"].between(1, 6))
    d["Income"] = raw["INCOME3"].where(raw["INCOME3"].between(1, 11)).clip(upper=8)  # 8–11 -> 8 ($75k+)

    feats = schema["feature_order"]
    lost = d.isna().sum().rename("dòng bị loại (thiếu / không biết / từ chối)")
    d = d.dropna()
    ok = np.ones(len(d), dtype=bool)
    for f in feats:
        r = schema["features"][f]
        ok &= d[f].between(r["min"], r["max"]).to_numpy()
    out_of_range = int((~ok).sum())
    d = d[ok].astype("int64")[["Diabetes_binary"] + feats]
    return d, lost, out_of_range


schema = json.load(open("deployment_bundle/feature_schema.json"))
raw = pd.concat(c[SRC] for c in pd.read_sas(XPT, format="xport", chunksize=50_000, encoding="latin-1"))
d25, lost, out_of_range = recode(raw, schema)

NEW = "/content/brfss2025_recoded.csv"
d25.to_csv(NEW, index=False)
print(f"BRFSS 2025 gốc: {len(raw):,} dòng → giữ lại {len(d25):,} dòng "
      f"({len(d25) / len(raw):.1%}) · tỉ lệ dương tính {d25.Diabetes_binary.mean():.2%}")
print(f"Ngoài khoảng giá trị của schema: {out_of_range:,} dòng")
display(lost.sort_values(ascending=False).to_frame())

### Kiểm tra recode (bắt buộc đọc)

Bảng dưới so giá trị trung bình từng cột giữa 2015 và 2025. Với cột 0/1, đây là tỉ lệ người trả lời "có". Hai năm nên gần nhau.

Nếu một cột bị **lật ngược** (ví dụ 2015 = 0.43, 2025 = 0.57), mã của biến đó bị đảo. Khi đó **dừng lại, chưa chạy ô 7**.

In [ ]:
# 6 · So sánh 2015 và 2025
cmp = pd.DataFrame({"2015": h[d25.columns].mean(), "2025": d25.mean()})
cmp["chênh lệch"] = cmp["2025"] - cmp["2015"]
binary = [c for c in d25.columns if set(d25[c].unique()) <= {0, 1}]
suspicious = [c for c in binary if abs(cmp.loc[c, "chênh lệch"]) > 0.15]
display(cmp.round(3))
print("⚠️ Cần xem lại:", suspicious) if suspicious else print("OK · không cột 0/1 nào lệch quá 0.15")

In [ ]:
# 7 · Chạy pipeline retrain (khoảng 5–15 phút)
#     Thêm --skip-cv để nhanh; bỏ --skip-cv ở lần chạy cuối nếu muốn có OOF PR-AUC.
!python retrain_pipeline.py --new-data {NEW} --ignore-schedule --skip-cv

In [ ]:
# 8 · Report đầy đủ
from IPython.display import Markdown
RUN = sorted(glob.glob("retraining_runs/*/"))[-1]
display(Markdown(open(RUN + "report.md", encoding="utf-8").read()))

In [ ]:
# 9 · Bảng gọn cho slide: model 2015 (v1.0.0) vs model retrain, trên cùng các dòng
rep = json.load(open(RUN + "report.json", encoding="utf-8"))
if "evaluation" not in rep:
    print("Không có kết quả đánh giá · STATUS:", rep["status"], rep.get("policy_problems"))
else:
    ev, rows = rep["evaluation"], []
    names = {"new_holdout": "2025 holdout (chưa model nào học)", "locked_test": "2015 locked test"}
    for s in ("new_holdout", "locked_test"):
        if s not in ev["candidate"]:
            continue
        p, c = ev["production"][s], ev["candidate"][s]
        for key, label in (("pr_auc", "PR-AUC"), ("roc_auc", "ROC-AUC"), ("brier", "Brier")):
            rows.append((f"{names[s]} · n={c['n']:,}", label, p[key], c[key]))
        for rp, rc in zip(p["capacity"], c["capacity"]):
            rows.append((f"{names[s]} · n={c['n']:,}", f"Precision@{rp['capacity_pct']}", rp["precision_at_k"], rc["precision_at_k"]))
    table = pd.DataFrame(rows, columns=["Tập đánh giá", "Chỉ số", "v1.0.0 (2015)", "Retrained (2015+2025)"])
    table["Δ"] = table["Retrained (2015+2025)"] - table["v1.0.0 (2015)"]
    display(table.round(4))
    print("STATUS:", rep["status"])
    for ch in rep["checks"]:
        print(("PASS " if ch["passed"] else "FAIL ") + ch["check"], f"({ch['value']:+.4f})")

In [ ]:
# 10 · Tải kết quả về máy (report + model ứng viên)
from google.colab import files
cand = rep.get("candidate_bundle")
!zip -qr /content/retrain_2025_result.zip {RUN} {cand if cand else ""}
files.download("/content/retrain_2025_result.zip")